# U-Net–ResNet34 — Microscopy Semantic Segmentation

**Research code · Dr. Gülsüm Meço**

Original research implementation of a U-Net with ResNet34 encoder, including semantic-mask loading, training, validation and checkpoint saving.

**Prerequisites:** Requires the same external COCO microscopy dataset and the semantic masks generated by the SegFormer preparation notebook at /content/own_polygonal_39_masks. Mount Google Drive if the configured Drive checkpoint path is used.

**Dataset classes:** background, onion, seed, stem. Dataset images, annotation files, and trained model weights are **not published** in this repository.

**Release note:** Adapted from the author's working Colab notebook without including its exploratory cells or execution outputs. This notebook preserves original experimental code but was not re-run end to end as part of the public-release preparation.


In [ ]:
# ============================================================
# OWN_POLYGONAL_39 — U-NET + RESNET34 RECONSTRUCTION
# ============================================================

!pip -q install segmentation-models-pytorch

import os, json, random
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
from torchvision.transforms import functional as TF

# ------------------------------------------------------------
# AYARLAR
# ------------------------------------------------------------
SEED = 42

BASE = "/content/own_polygonal_39"
MASK_BASE = "/content/own_polygonal_39_masks"

LOCAL_SAVE = "/content/best_unet_own_polygonal_39.pt"
DRIVE_SAVE = "/content/drive/MyDrive/own_polygonal_39_checkpoints/best_unet_own_polygonal_39.pt"

NUM_CLASSES = 4
CLASS_NAMES = ["background", "onion", "seed", "stem"]

BATCH_SIZE = 2
LR = 6e-5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 50
PATIENCE = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------
class UNetMicroscopyDataset(Dataset):

    def __init__(self, split):

        self.split = split
        self.img_dir = os.path.join(BASE, split)
        self.mask_dir = os.path.join(MASK_BASE, split)

        ann_path = os.path.join(
            self.img_dir,
            "_annotations.coco.json"
        )

        with open(ann_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        self.images = coco["images"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        info = self.images[idx]

        image_path = os.path.join(
            self.img_dir,
            info["file_name"]
        )

        mask_name = (
            os.path.splitext(info["file_name"])[0]
            + ".png"
        )

        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        # 512x512
        image = TF.resize(
            image,
            [512, 512],
            interpolation=TF.InterpolationMode.BILINEAR
        )

        mask = TF.resize(
            mask,
            [512, 512],
            interpolation=TF.InterpolationMode.NEAREST
        )

        image = TF.to_tensor(image)

        # ImageNet normalization
        image = TF.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        mask = torch.from_numpy(
            np.array(mask, dtype=np.int64)
        )

        return image, mask


train_ds = UNetMicroscopyDataset("train")
valid_ds = UNetMicroscopyDataset("valid")
test_ds  = UNetMicroscopyDataset("test")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Dataset:",
    len(train_ds),
    len(valid_ds),
    len(test_ds)
)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

print(
    "Parametre:",
    sum(p.numel() for p in model.parameters())
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

criterion = torch.nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------
@torch.no_grad()
def validation_metrics():

    model.eval()

    cm = torch.zeros(
        NUM_CLASSES,
        NUM_CLASSES,
        dtype=torch.int64
    )

    losses = []

    for x, y in valid_loader:

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(x)
            loss = criterion(logits, y)

        losses.append(loss.item())

        pred = logits.argmax(1)

        yt = y.reshape(-1).cpu()
        yp = pred.reshape(-1).cpu()

        valid = (
            (yt >= 0)
            & (yt < NUM_CLASSES)
        )

        indices = (
            NUM_CLASSES * yt[valid]
            + yp[valid]
        )

        cm += torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        ).reshape(NUM_CLASSES, NUM_CLASSES)

    dices = []

    for c in range(1, NUM_CLASSES):

        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp

        dice = (
            2 * tp /
            (2 * tp + fp + fn + 1e-7)
        )

        dices.append(dice)

    return (
        float(np.mean(losses)),
        float(np.mean(dices)),
        dices
    )

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------
best_dice = -1
best_epoch = -1
wait = 0

for epoch in range(1, MAX_EPOCHS + 1):

    model.train()

    train_losses = []

    for x, y in train_loader:

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())

    val_loss, val_dice, class_dices = validation_metrics()

    print(
        f"Epoch {epoch:02d} | "
        f"Train {np.mean(train_losses):.4f} | "
        f"Val {val_loss:.4f} | "
        f"FG Dice {val_dice:.6f} | "
        f"Onion {class_dices[0]:.4f} | "
        f"Seed {class_dices[1]:.4f} | "
        f"Stem {class_dices[2]:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice
        best_epoch = epoch
        wait = 0

        checkpoint = {
            "epoch": epoch,
            "best_val_dice": best_dice,
            "class_names": CLASS_NAMES,
            "encoder": "resnet34",
            "model_state_dict": model.state_dict()
        }

        torch.save(checkpoint, LOCAL_SAVE)
        torch.save(checkpoint, DRIVE_SAVE)

        print("  >>> BEST CHECKPOINT KAYDEDİLDİ")

    else:

        wait += 1

        if wait >= PATIENCE:
            print("\nEarly stopping.")
            break

print("\n================================")
print("U-NET TAMAMLANDI")
print("Best epoch:", best_epoch)
print("Best validation Dice:", best_dice)
print("Local:", LOCAL_SAVE)
print("Drive:", DRIVE_SAVE)
print("================================")

# ------------------------------------------------------------
# CHECKPOINTLERİ DOĞRULA
# ------------------------------------------------------------
print("\nDRIVE KONTROLÜ")

for p in [
    "/content/drive/MyDrive/own_polygonal_39_checkpoints/best_segformer_own_polygonal_39.pt",
    "/content/drive/MyDrive/own_polygonal_39_checkpoints/best_unet_own_polygonal_39.pt"
]:
    print(
        os.path.exists(p),
        round(os.path.getsize(p) / 1024**2, 2) if os.path.exists(p) else 0,
        "MB ->",
        p
    )